# 1. YouTube 字幕爬蟲（台中美食版）

改自組員的台北版 `youtube爬蟲.ipynb`：搜尋臺中美食影片，篩選年份與主題後，抓取**影片資訊＋字幕＋說明欄**（不下載影片），輸出 JSON 給 `2_餐廳抽取.ipynb`。

沒有字幕的影片（很多 YouTuber 把字幕壓在畫面上）在**步驟 5（選用）**只下載音訊，用 Whisper 在本機轉成文字（D017）。

> **輸出放在 `raw/`，不進 git**：逐字稿屬非官方取得，只供研究原型內部使用、不公開散布（D017），而群組 repo 是公開的。

## 事前安裝（Windows／Mac 都可）

| 項目 | 用途 | 安裝 |
|---|---|---|
| Python 3.11 以上 | 執行環境（`settings.py` 用到 `tomllib`） | VS Code 右上角「Select Kernel」選 3.11+ |
| yt-dlp、pandas、openpyxl | 爬蟲、匯出 | 步驟 0-1 |
| faster-whisper、opencc | 語音轉文字、簡轉繁 | 步驟 0-2（只有步驟 5 需要；faster-whisper 自帶音訊解碼，**不用另外裝 ffmpeg**） |

## 執行順序

| 步驟 | 內容 | 備註 |
|---|---|---|
| 0 | 安裝套件 | 只要跑一次 |
| 1 | 設定關鍵字、年份、指定影片 | 改設定後從這裡重跑 |
| 2 | 載入工具函式 | |
| 3 | 搜尋影片清單 | 會清空先前的結果，重新開始 |
| 4 | 抓影片資訊＋字幕 | 中斷後重跑會從斷點繼續 |
| 5 | （選用）沒字幕的影片 → 音訊 → Whisper | 約影片長度的 1/5 |
| 6 | 清洗並匯出 CSV（給 Excel）＋ JSON（給 `2_餐廳抽取`） | |


## 步驟 0：安裝套件

In [ ]:
# 0-1 必裝
%pip install -q -U yt-dlp pandas openpyxl

In [ ]:
# 0-2 選裝：要跑步驟 5（語音轉文字）才需要
%pip install -q faster-whisper opencc-python-reimplemented

## 步驟 1：設定

- `CHANNELS` 留空 `[]` → 用 `KEYWORDS` 搜尋全 YouTube；填頻道則在每個頻道各搜一次
- `EXTRA_VIDEO_IDS`：手動指定的影片，和搜尋結果一起處理（一樣會套用年份等篩選）
- `RELEVANT_WORDS`：標題或標籤要出現其中一個詞才收錄
- 挑選標準（頻道、數量、年份、區域）尚待全組確認（DESIGN.md O001），以下是暫定值


In [ ]:
CHANNELS = []                                      # [] = 搜尋全 YouTube
KEYWORDS = ["台中美食", "台中約會餐廳", "台中慶生餐廳", "台中聚餐餐廳",
            "台中火車站美食", "勤美美食", "逢甲美食", "一中街美食"]
EXTRA_VIDEO_IDS = ["N5xxdvKyqxI", "gJhCCxoPsmU"]   # 手動指定（勤美草悟道 Top12、台中火車站老字號小吃）
MAX_VIDEOS_PER_QUERY = 10                          # 每次搜尋最多取幾支

START_YEAR, END_YEAR = 2023, 2026                  # 暫定（O001）；gJhCCxoPsmU 是 2021 年，會被這條濾掉
RELEVANT_WORDS = ["台中", "臺中", "Taichung"]       # 標題或標籤要含其中一個；[] = 不篩選
EXCLUDE_WORDS = ["看房", "豪宅", "租屋", "套房", "房價"]  # 標題含這些詞就略過
MIN_DURATION = 60                                  # 排除 Shorts
SKIP_NO_SUBTITLE = False                           # False = 保留，可在步驟 5 用 Whisper 補
SUB_LANGS = ["zh-TW", "zh-Hant", "zh", "zh-HK", "zh-Hans", "zh-CN"]   # 字幕語言優先順序（不收英文軌）

OUTPUT_DIR = "raw/youtube"
OUTPUT_NAME = "youtube_taichung_food"              # 會產生 .json / _cleaned.json / _cleaned.csv

## 步驟 2：載入工具函式

In [ ]:
import os
import json
import time
import random
from datetime import datetime
from urllib.parse import quote

import pandas as pd
import yt_dlp

# node 當 JS runtime：沒有的話新版 yt-dlp 會警告、部分格式抓不到（Windows 裝了 Node.js 即可）
YDL_OPTS = {"quiet": True, "no_warnings": True, "skip_download": True, "js_runtimes": {"node": {}}}
JSON_PATH = os.path.join(OUTPUT_DIR, OUTPUT_NAME + ".json")


def polite_sleep(a=2, b=4):
    """隨機休息，放慢速度避免被擋"""
    time.sleep(random.uniform(a, b))


def search_video_ids(keyword, channel=None, limit=MAX_VIDEOS_PER_QUERY):
    """搜尋影片 ID（只拿清單，不抓細節，速度快）；keyword 為 None 時抓頻道最新影片"""
    if channel and keyword is None:
        url = f"{channel.rstrip('/')}/videos"
    elif channel:
        url = f"{channel.rstrip('/')}/search?query={quote(keyword)}"
    else:
        url = f"ytsearch{limit}:{keyword}"
    with yt_dlp.YoutubeDL({**YDL_OPTS, "extract_flat": True, "playlistend": limit}) as ydl:
        info = ydl.extract_info(url, download=False)
    # 頻道搜尋結果會混入播放清單，只留影片（ID 長度 11）
    return [e["id"] for e in info.get("entries") or [] if e and len(e.get("id", "")) == 11][:limit]


def skip_reason(info):
    """回傳不收錄的原因；符合條件回傳 None"""
    upload = info.get("upload_date") or ""
    if not upload or not (START_YEAR <= int(upload[:4]) <= END_YEAR):
        return f"{upload[:4] or '無日期'} 年"
    if (info.get("duration") or 0) < MIN_DURATION:
        return f"太短 {info.get('duration')} 秒"
    title = info.get("title") or ""
    hit = next((w for w in EXCLUDE_WORDS if w in title), None)
    if hit:
        return f"排除詞「{hit}」"
    if RELEVANT_WORDS:
        haystack = title + " " + " ".join(info.get("tags") or [])
        if not any(w.lower() in haystack.lower() for w in RELEVANT_WORDS):
            return "標題/標籤不相關"
    return None


def pick_subtitle_track(info):
    """挑字幕：人工字幕 > 自動字幕原始語言 (-orig) > 自動字幕的偏好語言（機器翻譯）"""
    subs = info.get("subtitles") or {}
    for lang in SUB_LANGS:
        if lang in subs:
            return "人工字幕", lang, subs[lang]
    auto = info.get("automatic_captions") or {}
    orig = [k for k in auto if k.endswith("-orig") and k.startswith("zh")]
    if orig:
        return "自動字幕", orig[0].replace("-orig", ""), auto[orig[0]]
    for lang in SUB_LANGS:
        if lang in auto:
            return "自動字幕(翻譯)", lang, auto[lang]
    return "無字幕", "", None


def fetch_subtitle_text(ydl, formats):
    """直接讀取 json3 字幕內容（不存檔），回傳整段文字"""
    fmt = next((f for f in formats if f.get("ext") == "json3"), None)
    if fmt is None:
        return ""
    data = json.loads(ydl.urlopen(fmt["url"]).read().decode("utf-8"))
    lines = []
    for ev in data.get("events", []):
        text = "".join(s.get("utf8", "") for s in ev.get("segs") or []).strip()
        if text and (not lines or text != lines[-1]):   # 自動字幕常重複上一行
            lines.append(text)
    return "\n".join(lines)


def save_progress():
    """把目前結果存成 JSON（步驟 4、5 會邊跑邊存，中斷也不會全部遺失）"""
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(list(collected.values()), f, ensure_ascii=False, indent=1)

print("工具函式載入完成")

## 步驟 3：搜尋影片清單

會清空先前的結果重新開始。如果 kernel 重開過、想接續上次存的 JSON，把 `RESUME_FROM_FILE` 改成 `True`。

In [ ]:
RESUME_FROM_FILE = False

collected = {}   # 影片ID → 資料
skipped = {}     # 影片ID → 略過原因
if RESUME_FROM_FILE and os.path.exists(JSON_PATH):
    with open(JSON_PATH, encoding="utf-8") as f:
        collected = {r["影片ID"]: r for r in json.load(f)}
    print(f"從 {JSON_PATH} 接續，已有 {len(collected)} 支")

if CHANNELS:
    queries = [(ch, kw) for ch in CHANNELS for kw in (KEYWORDS or [None])]
else:
    queries = [(None, kw) for kw in KEYWORDS]

video_ids = {v: "手動指定" for v in EXTRA_VIDEO_IDS}   # 影片ID → 搜尋關鍵字（保留順序、去重）
failed_queries = []  # 抓取失敗的 (頻道, 關鍵字)
for channel, keyword in queries:
    label = f"在 {channel or '全 YouTube'} " + (f"搜尋「{keyword}」" if keyword else "抓最新影片")
    try:
        ids = search_video_ids(keyword, channel)
    except yt_dlp.utils.DownloadError as e:
        print(f"[警告] {label}失敗，略過：{e}")
        failed_queries.append((channel, keyword))
        continue
    new = [v for v in ids if v not in video_ids]
    video_ids.update({v: keyword or "頻道最新影片" for v in new})
    print(f"{label}：找到 {len(ids)} 支，新增 {len(new)} 支")
    polite_sleep(1, 2)

print(f"\n共 {len(video_ids)} 支影片待處理（含手動指定 {len(EXTRA_VIDEO_IDS)} 支）")
if failed_queries:
    print(f"有 {len(failed_queries)} 次搜尋失敗：{failed_queries}")

## 步驟 4：抓影片資訊＋字幕

依年份、長度、`RELEVANT_WORDS` 篩選後，抓取字幕文字與**原始說明欄**（店名、地址、營業時間多半寫在這裡）。中斷後重跑這格，會從斷點繼續。

In [ ]:
todo = [v for v in video_ids if v not in collected and v not in skipped]
print(f"這次要處理 {len(todo)} 支（已收錄 {len(collected)}、已略過 {len(skipped)}）\n")

try:
    with yt_dlp.YoutubeDL(YDL_OPTS) as ydl:
        for vid in todo:
            url = f"https://www.youtube.com/watch?v={vid}"
            try:
                info = ydl.extract_info(url, download=False)
            except yt_dlp.utils.DownloadError as e:
                print(f"[警告] 無法取得影片 {url}: {e}")
                skipped[vid] = "無法取得"
                continue

            reason = skip_reason(info)
            if reason:
                skipped[vid] = reason
                print(f"  略過（{reason}）：{info.get('title')}")
                continue

            source, lang, formats = pick_subtitle_track(info)
            text = fetch_subtitle_text(ydl, formats) if formats else ""
            if not text and SKIP_NO_SUBTITLE:
                skipped[vid] = "無字幕"
                print(f"  略過（無字幕）：{info.get('title')}")
                continue

            collected[vid] = {
                "影片ID": vid,
                "頻道": info.get("channel"),
                "搜尋關鍵字": video_ids[vid],
                "影片標題": info.get("title"),
                "上傳日期": datetime.strptime(info["upload_date"], "%Y%m%d").strftime("%Y-%m-%d"),
                "取得時間": datetime.now().isoformat(timespec="seconds"),
                "影片長度(秒)": info.get("duration"),
                "觀看次數": info.get("view_count"),
                "按讚數": info.get("like_count"),
                "標籤": ",".join(info.get("tags") or []),
                "字幕來源": source if text else "無字幕",
                "字幕語言": lang if text else "",
                "影片網址": url,
                "影片說明": info.get("description") or "",
                "字幕內容": text,
            }
            save_progress()
            print(f"成功收錄 [{len(collected) - 1}] ({collected[vid]['字幕來源']}): {info.get('title')}")
            polite_sleep()

except Exception as e:
    print(f"\n[錯誤] 爬取中斷（已存檔，重跑這格可繼續）: {e}")
    if "429" in str(e):
        print("  → YouTube 暫時限流：等 30–60 分鐘，把步驟 3 的 RESUME_FROM_FILE 改成 True，從步驟 3 重跑")

no_sub = sum(1 for r in collected.values() if not r["字幕內容"])
print(f"\n--- 完成 ---\n共收錄 {len(collected)} 支，有字幕 {len(collected) - no_sub} 支、無字幕 {no_sub} 支；略過 {len(skipped)} 支")
if collected:
    print("\n各頻道收錄數：")
    print(pd.Series([r["頻道"] for r in collected.values()]).value_counts().to_string())

## 步驟 5（選用）：沒字幕的影片 → 下載音訊 → Whisper 語音轉文字

- 只下載音訊（不下載影片），轉完文字就刪掉音訊檔
- 在本機跑，免費；`small` 模型約需影片長度的 1/5（第一次會下載約 500MB 模型）
- 每轉完一支就存檔，中斷後重跑這格會跳過已完成的
- 要重轉某支（例如自動字幕品質太差），把它的 `字幕內容` 清空再跑這格

In [ ]:
WHISPER_MODEL = "small"     # tiny / base / small / medium / large-v3-turbo（越大越準、越慢）
ASR_MAX_VIDEOS = 10         # 這次最多轉幾支；None = 全部
AUDIO_DIR = os.path.join(OUTPUT_DIR, "audio")

from faster_whisper import WhisperModel
from opencc import OpenCC

to_traditional = OpenCC("s2twp").convert   # 簡體 → 台灣繁體用語

if not collected and os.path.exists(JSON_PATH):    # kernel 重開過：從 JSON 讀回
    with open(JSON_PATH, encoding="utf-8") as f:
        collected = {r["影片ID"]: r for r in json.load(f)}


def download_audio(vid):
    opts = {**YDL_OPTS, "skip_download": False, "noprogress": True, "format": "ba[ext=m4a]/ba",
            "outtmpl": os.path.join(AUDIO_DIR, "%(id)s.%(ext)s")}
    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(f"https://www.youtube.com/watch?v={vid}", download=True)
        return ydl.prepare_filename(info)


if "whisper" not in globals():
    print(f"載入 Whisper 模型 {WHISPER_MODEL}（第一次會下載）...")
    whisper = WhisperModel(WHISPER_MODEL, device="cpu", compute_type="int8")

targets = [r for r in collected.values() if not r["字幕內容"]][:ASR_MAX_VIDEOS]
print(f"這次要轉 {len(targets)} 支\n")

for i, row in enumerate(targets, 1):
    start = time.time()
    print(f"[{i}/{len(targets)}] {row['影片標題']}（{(row['影片長度(秒)'] or 0) // 60} 分鐘）")
    try:
        path = download_audio(row["影片ID"])
        # faster-whisper 直接吃檔案路徑（內建 PyAV 解碼），不需要 ffmpeg
        segments, _ = whisper.transcribe(path, language="zh", vad_filter=True, beam_size=5,
                                         initial_prompt="以下是台灣YouTuber的台中美食影片，使用繁體中文。")
        row["字幕內容"] = "\n".join(to_traditional(s.text.strip()) for s in segments)
        row["字幕來源"] = f"Whisper語音辨識({WHISPER_MODEL})"
        row["字幕語言"] = "zh"
        save_progress()
        os.remove(path)
        print(f"    完成，花了 {time.time() - start:.0f} 秒")
    except Exception as e:
        print(f"    [警告] 失敗：{e}")

left = sum(1 for r in collected.values() if not r["字幕內容"])
print(f"\n--- 完成 --- 還有 {left} 支沒有文字，可再執行這格繼續")

## 步驟 6：清洗並匯出

清洗規則：移除音效標記（[音樂]、(笑)…）、網址、hashtag、分隔線。

| 檔案 | 用途 |
|---|---|
| `_cleaned.csv` | 用 Excel 檢視（換行換成空白；Excel 單格上限 32,767 字，過長會截斷） |
| `_cleaned.json` | 給 `2_餐廳抽取` 用：字幕**保留換行**（每行一句）；另附 `影片說明(原始)`，因為清洗會刪掉說明欄的 `#店名` |


In [ ]:
import re


def clean_subtitle_text(text, keep_newlines=False):
    """清洗 YouTube 字幕／說明。keep_newlines=True 保留換行（給抽取用），否則全部換成空白（給 Excel 用）"""
    if not isinstance(text, str):
        return ""
    text = re.sub(r'[\[［(（](音樂|音乐|Music|掌聲|掌声|Applause|笑|笑聲|Laughter)[\]］)）]', '', text, flags=re.I)
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'#\S+', '', text)
    text = re.sub(r'[-=＿_─━]{3,}', ' ', text)
    if keep_newlines:
        lines = (re.sub(r'[ \t\u3000]+', ' ', line).strip() for line in text.splitlines())
        return "\n".join(line for line in lines if line)
    return re.sub(r'\s+', ' ', text).strip()


with open(JSON_PATH, encoding='utf-8') as f:
    rows = json.load(f)
df = pd.DataFrame(rows)

csv_df = df.copy()
csv_df['字幕內容'] = csv_df['字幕內容'].apply(clean_subtitle_text)
csv_df['影片說明'] = csv_df['影片說明'].apply(clean_subtitle_text)
csv_path = os.path.join(OUTPUT_DIR, OUTPUT_NAME + "_cleaned.csv")
csv_df.to_csv(csv_path, index=False, encoding='utf-8-sig')

clean_rows = [dict(r, 字幕內容=clean_subtitle_text(r.get('字幕內容'), keep_newlines=True),
                   影片說明=clean_subtitle_text(r.get('影片說明'), keep_newlines=True),
                   **{"影片說明(原始)": r.get('影片說明') or ""}) for r in rows]
clean_json_path = os.path.join(OUTPUT_DIR, OUTPUT_NAME + "_cleaned.json")
with open(clean_json_path, 'w', encoding='utf-8') as f:
    json.dump(clean_rows, f, ensure_ascii=False, indent=1)

print(f"資料清洗完成：\n  CSV （給 Excel）：{csv_path}\n  JSON（給 2_餐廳抽取）：{clean_json_path}")
print(f"共 {len(df)} 筆，其中 {(csv_df['字幕內容'] != '').sum()} 筆有文字。")